# Quantizing a Tiny Transformer LLM from Scratch

> **Week 11 Lecture Demo** — CS 203: Software Tools and Techniques for AI · IIT Gandhinagar

In notebook `09-quantization-from-scratch.ipynb` we quantized a tiny MLP on the two-moons dataset. Now we do the same thing on a **real transformer language model** — one we train from scratch in this notebook, with no HuggingFace, no `torch.quantization`, and no downloads. The whole pipeline sits in ~100 lines.

Plan:
1. Build a **character-level tokenizer** from a small hardcoded Shakespeare snippet.
2. Define a **tiny decoder-only transformer** (≈20K–80K parameters, 2 layers).
3. Train it for a few hundred steps on CPU.
4. Generate text in FP32.
5. **Quantize every `nn.Linear` in the model to INT8 by hand** (three lines of arithmetic, same as the MLP notebook).
6. Generate text in "INT8" and compare: output text, per-token loss, and model size.

You will see that INT8 generation stays *almost* identical to FP32 — with 4× less memory for the linear weights.


In [ ]:
import math
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1)
np.random.seed(1)

device = "cpu"  # keep everything CPU-friendly


---

## 1. A tiny hardcoded corpus

No downloads. Just the opening of Hamlet's soliloquy, plus a bit more — enough to give a character-level model some structure to learn.


In [ ]:
CORPUS = """To be, or not to be: that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die: to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause: there's the respect
That makes calamity of so long life;
For who would bear the whips and scorns of time,
The oppressor's wrong, the proud man's contumely,
The pangs of despised love, the law's delay,
The insolence of office and the spurns
That patient merit of the unworthy takes,
When he himself might his quietus make
With a bare bodkin? who would fardels bear,
To grunt and sweat under a weary life,
But that the dread of something after death,
The undiscover'd country from whose bourn
No traveller returns, puzzles the will
And makes us rather bear those ills we have
Than fly to others that we know not of?
Thus conscience does make cowards of us all;
And thus the native hue of resolution
Is sicklied o'er with the pale cast of thought,
And enterprises of great pith and moment
With this regard their currents turn awry,
And lose the name of action."""

print(f"corpus length: {len(CORPUS)} characters")
print(f"first 120 chars: {CORPUS[:120]!r}")


### Character-level tokenizer

The vocabulary is just the unique characters in the corpus. `stoi` maps char → id, `itos` maps back.


In [ ]:
chars = sorted(set(CORPUS))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}

def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(itos[int(i)] for i in ids)

data = torch.tensor(encode(CORPUS), dtype=torch.long)
print(f"vocab size: {vocab_size}")
print(f"tokenized length: {len(data)}")
print(f"sample round-trip: {decode(encode('To be'))!r}")


---

## 2. A tiny decoder-only transformer

- 2 transformer blocks
- `d_model = 48`, `n_head = 4` (head dim = 12)
- 4× FFN (as in GPT)
- context length 64

We write our own `CausalSelfAttention` with four **explicit `nn.Linear` layers** (`q`, `k`, `v`, `proj`) instead of `nn.MultiheadAttention` — that way our quantizer can treat them uniformly.


In [ ]:
CTX = 64

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, mask):
        b, t, d = x.shape
        q = self.q(x).view(b, t, self.n_head, self.d_head).transpose(1, 2)
        k = self.k(x).view(b, t, self.n_head, self.d_head).transpose(1, 2)
        v = self.v(x).view(b, t, self.n_head, self.d_head).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att + mask[:t, :t]
        att = F.softmax(att, dim=-1)
        out = (att @ v).transpose(1, 2).contiguous().view(b, t, d)
        return self.proj(out)


class Block(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head)
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)
        h = self.fc2(F.gelu(self.fc1(self.ln2(x))))
        return x + h


class TinyLLM(nn.Module):
    def __init__(self, vocab_size, d_model=48, n_layer=2, n_head=4, ctx=CTX):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(ctx, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.ctx = ctx
        self.register_buffer(
            "mask", torch.triu(torch.full((ctx, ctx), float('-inf')), diagonal=1)
        )

    def forward(self, idx):
        b, t = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(t, device=idx.device))
        for blk in self.blocks:
            x = blk(x, self.mask)
        return self.head(self.ln_f(x))

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, greedy=False):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.ctx:]
            logits = self(idx_cond)[:, -1, :] / temperature
            if greedy:
                idx_next = logits.argmax(-1, keepdim=True)
            else:
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, 1)
            idx = torch.cat([idx, idx_next], dim=1)
        return idx


model = TinyLLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"model parameters: {n_params:,}")


---

## 3. Train for a few hundred steps on CPU

We sample random `(CTX+1)`-length windows from the corpus and train next-token prediction with cross-entropy.


In [ ]:
def get_batch(batch_size=16):
    ix = torch.randint(0, len(data) - CTX - 1, (batch_size,))
    x = torch.stack([data[i : i + CTX] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + CTX] for i in ix])
    return x.to(device), y.to(device)


opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
model.train()

for step in range(600):
    x, y = get_batch(32)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if (step + 1) % 100 == 0:
        print(f"step {step+1:4d}  loss={loss.item():.4f}")

model.eval()


### Sample from the FP32 model


In [ ]:
torch.manual_seed(42)
prompt = "To be, or"
ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out = model.generate(ids, max_new_tokens=200, temperature=0.8)
fp32_text = decode(out[0].tolist())
print(fp32_text)


The generation won't be fluent Shakespeare — we only trained for 600 steps on a 48-dim model over a 1.5 KB corpus — but it should look roughly **English-ish**: plausible letter n-grams, something resembling words, line breaks in sensible places. Good enough for a quantization demo.


---

## 4. INT8 symmetric quantization — applied to every `nn.Linear`

Same three-line arithmetic as the MLP notebook. We walk every `nn.Linear` in the model, compute `scale = max(|W|) / 127`, store `q = round(W/scale)` as `int8`, and keep the `bias` in FP32.


In [ ]:
def quantize_tensor(x):
    """Symmetric per-tensor INT8 quantization."""
    max_abs = x.abs().max().item()
    if max_abs == 0:
        return torch.zeros_like(x, dtype=torch.int8), 1.0
    scale = max_abs / 127.0
    q = torch.round(x / scale).clamp(-127, 127).to(torch.int8)
    return q, scale


def dequantize_tensor(q, scale):
    return q.to(torch.float32) * scale


def quantize_model(net):
    """Replace every nn.Linear.weight with a fake-quantized FP32 tensor.
    Returns a deep-copied net so the original FP32 model is unchanged."""
    qnet = copy.deepcopy(net)
    report = []
    for name, m in qnet.named_modules():
        if isinstance(m, nn.Linear):
            q, s = quantize_tensor(m.weight.data)
            m.weight.data = dequantize_tensor(q, s)         # "fake quantize"
            m._int8_weight = q                              # keep the int8 values
            m._scale = s
            err = (net.get_submodule(name).weight.data - m.weight.data).abs().max().item()
            report.append((name, tuple(q.shape), s, err))
    return qnet, report


qmodel, report = quantize_model(model)

print(f"{'layer':40s} {'shape':14s} {'scale':>12s} {'max_err':>10s}")
for name, shape, scale, err in report:
    print(f"{name:40s} {str(shape):14s} {scale:12.6f} {err:10.6f}")


### Generate from the INT8 model with the same random seed

Because we replaced the weights in-place with their dequantized versions, we can simply call `.generate()` on `qmodel` — no custom forward pass needed. Same sampling seed = apples-to-apples comparison.


In [ ]:
torch.manual_seed(42)
ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
out_q = qmodel.generate(ids, max_new_tokens=200, temperature=0.8)
int8_text = decode(out_q[0].tolist())
print(int8_text)


### Side-by-side


In [ ]:
print("FP32\n" + "-" * 60)
print(fp32_text)
print("\nINT8\n" + "-" * 60)
print(int8_text)


Read the two carefully. For the first ~20–40 tokens they will typically be **character-for-character identical** (the quantization error hasn't accumulated into a different `argmax` yet). After that they diverge stochastically because sampling is chaotic — but *both* continue to produce the same style of near-English gibberish. No collapse, no gibberish-squared.


---

## 5. Quantitative comparison

### 5a. Next-token loss on the training corpus

Same inputs, same targets, both models in eval mode. If quantization is working correctly, the loss shouldn't move much.


In [ ]:
@torch.no_grad()
def avg_loss(net, n_batches=32, batch_size=32):
    net.eval()
    total = 0.0
    for _ in range(n_batches):
        x, y = get_batch(batch_size)
        logits = net(x)
        total += F.cross_entropy(logits.view(-1, vocab_size), y.view(-1)).item()
    return total / n_batches


torch.manual_seed(7)
l_fp32 = avg_loss(model)
torch.manual_seed(7)
l_int8 = avg_loss(qmodel)
print(f"FP32 avg next-token CE loss: {l_fp32:.4f}")
print(f"INT8 avg next-token CE loss: {l_int8:.4f}")
print(f"relative change:             {(l_int8 - l_fp32) / l_fp32 * 100:+.2f}%")


### 5b. Model size

FP32 linear weights are 4 bytes/param; INT8 are 1 byte/param plus one FP32 scale per layer.


In [ ]:
def count_linear_bytes(net, dtype="fp32"):
    total = 0
    for m in net.modules():
        if isinstance(m, nn.Linear):
            total += m.weight.numel() * (4 if dtype == "fp32" else 1)
            if dtype == "int8":
                total += 4  # one scale per layer
            if m.bias is not None:
                total += m.bias.numel() * 4  # biases always FP32 in this scheme
    return total


def count_nonlinear_bytes(net):
    # embeddings, LayerNorm affine params, etc. — kept FP32 here
    total = 0
    for m in net.modules():
        if isinstance(m, nn.Linear):
            continue
        for p in m.parameters(recurse=False):
            total += p.numel() * 4
    return total


lin_fp32 = count_linear_bytes(model, "fp32")
lin_int8 = count_linear_bytes(model, "int8")
nonlin   = count_nonlinear_bytes(model)

print(f"Linear weights (FP32): {lin_fp32:>8,} bytes")
print(f"Linear weights (INT8): {lin_int8:>8,} bytes   -> {lin_fp32/lin_int8:.2f}x smaller")
print(f"Non-Linear params:     {nonlin:>8,} bytes   (embeddings, LayerNorm — kept FP32)")
print()
print(f"Total FP32 model: {lin_fp32 + nonlin:,} bytes")
print(f"Total INT8 model: {lin_int8 + nonlin:,} bytes   -> overall {(lin_fp32+nonlin)/(lin_int8+nonlin):.2f}x smaller")


### 5c. Per-layer quantization error

Which layers suffer the most rounding error? (Useful for deciding which ones to *keep* in FP32 if you were pushing for accuracy.)


In [ ]:
import matplotlib.pyplot as plt
names  = [r[0] for r in report]
errors = [r[3] for r in report]
plt.figure(figsize=(8, 4))
plt.bar(range(len(names)), errors)
plt.xticks(range(len(names)), names, rotation=75, ha="right", fontsize=7)
plt.ylabel("max |W - dequant(q)|")
plt.title("Per-layer round-trip quantization error")
plt.tight_layout(); plt.show()


The bars are essentially bounded by `scale/2` for each layer — exactly what you'd expect from rounding to the nearest int8 step. Layers with wider weight ranges have larger scales, hence larger per-weight error.


---

## 6. Exercises

1. **Quantize the embeddings too.** The `tok` and `pos` `nn.Embedding` tables are also just weight matrices. Extend `quantize_model` to handle them. Does loss move?

2. **Per-channel quantization for Linear.** One scale per *output row* of each weight matrix (dim=0). Does next-token loss improve vs the per-tensor scheme above?

3. **Quantization-aware head.** Keep the `head` Linear (which produces the vocab logits) in FP32 — quantize only the blocks. Does that move loss by much? What about keeping only the attention layers FP32?

4. **INT4 on this LLM.** Change `clamp(-127, 127)` to `clamp(-7, 7)` and re-run. At what model size does INT4 start to generate obvious garbage? Is the corpus you picked too small to notice?

5. **Static activation quantization.** Run one calibration pass over the training data; for the output of each `nn.Linear`, record `max|x|`. In a quantized forward pass, quantize those activations to INT8 at those breakpoints. How does loss change? (This is how real INT8 transformer inference engines like llama.cpp and TensorRT actually work.)

---

## What you learned

You just quantized a **real transformer language model** — embeddings, attention, FFN, LayerNorm, head — to INT8 using nothing but `round` and `clamp`. The whole quantization routine is under 20 lines.

Production INT8 LLM inference (`llama.cpp`, `GPTQ`, `bitsandbytes`, `AWQ`, etc.) is doing exactly this arithmetic, just with smarter calibration (GPTQ), per-channel + per-group grouping (GGUF), outlier handling (AWQ / SmoothQuant), and hand-written INT8 matmul kernels. But the core idea — **weights don't need 32 bits; 8 is usually enough** — is what you just demonstrated on a model you trained yourself.
